# Ledoit-Wolf

## Covariance Estimation

The obvious estimator from $T$ observations of $n$ assets is the sample covariance:

$$
\hat{\Sigma} = \frac{1}{T-1}\sum_{t=1}^T (r_t - \bar{r})(r_t - \bar{r})^T = \frac{1}{T-1}X^TX
$$

(after demeaning). It's unbiased and maximum-likelihood. So what's wrong with it? Three things, all of which you've now met from different angles.

In [1]:
import numpy as np
from sklearn.covariance import ledoit_wolf
from util_yahoo_finance import get_returns

**1. Rank deficiency when $T < n$ (the $n<p$ problem).** The rank of $\hat\Sigma$ is at most $\min(T-1, n)$. With 250 days and 500 assets, the $500\times500$ matrix has rank ≤ 249 — at least 251 eigenvalues are exactly zero. It's singular, non-invertible, not positive *definite*. Every optimizer that needs $\Sigma^{-1}$ (Markowitz, BL, risk parity) breaks. This is the futures-book covariance problem you lived.

**2. Eigenvalue dispersion even when $T > n$.** Even with enough data to be invertible, the sample covariance's eigenvalues are **systematically distorted by noise**: the largest are biased *too large*, the smallest biased *too small*. The matrix looks more "spread out" in its eigenvalue spectrum than the truth. And since the optimizer loads on the smallest eigenvalues (via $\Sigma^{-1}$, as you saw in the Markowitz-failures session), it's amplifying exactly the most-distorted, noisiest directions. Tomorrow's RMT makes this dispersion precise.

**3. Estimation error scales with dimension.** You're estimating $n(n+1)/2$ parameters — for 500 assets that's ~125,000 covariances — from $nT$ data points. When parameters ≈ data points, every entry is noisy, and those errors compound when you invert. The sample covariance is a *high-variance* estimator: low bias (it's unbiased), but enormous variance in high dimensions.

In [2]:
tickers = ["NVDA", "AAPL", "MSFT", "MU", "AMZN", "AMD", "GOOGL", "TSLA", "GOOG", "AVGO"]
returns = get_returns(tickers, start="2024-01-01")
cov = returns.cov().values * 252

rank = np.linalg.matrix_rank(cov)
print(f"Rank of the matrix: {rank}")

# We use eigvalsh instead of eigvals because it is highly optimized for symmetric matrices and guarantees real-number outputs.
eigenvalues = np.linalg.eigvalsh(cov)
print("Eigenvalues:", eigenvalues)

cond_num = np.linalg.cond(cov)
print(f"Condition Number: {cond_num}")

Rank of the matrix: 10
Eigenvalues: [2.16522957e-04 3.14746052e-02 4.49182079e-02 6.15942114e-02
 9.10694275e-02 1.34935919e-01 1.58094585e-01 1.70659844e-01
 2.62235930e-01 1.04387522e+00]
Condition Number: 4821.083349024155


The condition number tells you how much relative error in Σ gets amplified in Σ⁻¹. If your covariance is estimated from T=500 days, each entry has sampling noise of order 1/√T ≈ 4%. Σ⁻¹ amplifies that by up to κ = 500×, so the weights you compute from the inverse could absorb 500 × 4% = 2000% relative error from pure sampling noise. The optimizer is doing exactly what you asked — it just happens that you asked it to trust a noisy matrix, and it magnifies the noise.

**The bias-variance framing — this is the key to everything that follows.** The sample covariance is unbiased but high-variance. The structured alternatives we're about to introduce are biased but low-variance. The whole game of covariance estimation is trading a little bias for a large reduction in variance — accepting a "wrong" but stable matrix over a "right on average" but wildly unstable one. That's the bias-variance tradeoff from your stats foundation, now applied to matrix estimation, and it's exactly what shrinkage operationalizes.

## Shrinkage

Shrinkage operationalizes the bias-variance tradeoff directly. The recipe: blend the noisy sample covariance with a structured, low-variance **target**.

$$
\hat{\Sigma}_{\text{shrink}} = \delta\, F + (1 - \delta)\, \hat{\Sigma}
$$

- $\hat{\Sigma}$ — the sample covariance: unbiased, high-variance
- $F$ — the **shrinkage target**: a structured matrix, biased but low-variance
- $\delta \in [0,1]$ — the **shrinkage intensity**: how much to pull toward the target

You're literally taking a weighted average of "right on average but wild" and "wrong but stable." The result has a little bias and far less variance — and crucially, it's always invertible and well-conditioned if $F$ is.

**Why this directly fixes the problems:**

- **Rank/invertibility** — even if $\hat\Sigma$ is singular ($n<p$), the blend $\delta F + (1-\delta)\hat\Sigma$ is full-rank and invertible as long as $F$ is (and the identity, constant-correlation, and factor targets all are). The optimizer no longer hangs.
- **Eigenvalue dispersion** — shrinking pulls the over-dispersed sample eigenvalues back toward the target's (the large ones down, the small ones up), de-noising exactly the directions $\Sigma^{-1}$ would have amplified.

Shrinkage leaves one question: how much to shrink — what's the right $\delta$? Ledoit and Wolf's celebrated contribution is an **analytical, data-driven optimal $\delta$**, computed in closed form with no cross-validation, no tuning, no simulation.

**The common targets** (each encodes a different structural assumption):

1. **Scaled identity** $F = \frac{\text{tr}(\hat\Sigma)}{n}I$ — assume all assets have equal variance and zero correlation. Maximum structure, maximum bias. This is the original Ledoit-Wolf (2004) target.


In [3]:
n = cov.shape[0]
scaled_identity_f = (np.trace(cov) / n) * np.eye(n)
print(np.round(scaled_identity_f, 4))

[[0.1999 0.     0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.     0.1999 0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.     0.     0.1999 0.     0.     0.     0.     0.     0.     0.    ]
 [0.     0.     0.     0.1999 0.     0.     0.     0.     0.     0.    ]
 [0.     0.     0.     0.     0.1999 0.     0.     0.     0.     0.    ]
 [0.     0.     0.     0.     0.     0.1999 0.     0.     0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.1999 0.     0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.     0.1999 0.     0.    ]
 [0.     0.     0.     0.     0.     0.     0.     0.     0.1999 0.    ]
 [0.     0.     0.     0.     0.     0.     0.     0.     0.     0.1999]]


2. **Constant correlation** $F$ — keep each asset's own sample variance, but replace all pairwise correlations with the *average* sample correlation. Usually the best practical target for equities, because the dominant structure in a stock correlation matrix really is "everything is positively correlated by roughly the same amount" (the market factor). This is Ledoit-Wolf (2003b).

In [4]:
def constant_correlation_target(cov):
    """
    Ledoit-Wolf constant-correlation shrinkage target.

    Keeps each asset's own sample variance on the diagonal, but replaces
    every pairwise correlation with the grand average off-diagonal correlation.

        F_ii = sigma_i^2
        F_ij = rho_bar * sigma_i * sigma_j   (i != j)
    """
    std = np.sqrt(np.diag(cov))               # (n,) sample std devs
    corr = cov / np.outer(std, std)           # sample correlation matrix

    n = cov.shape[0]
    # average of all off-diagonal correlations
    rho_bar = (corr.sum() - n) / (n * (n - 1))

    # build target: start from rho_bar everywhere, then fix diagonal to 1
    F_corr = np.full((n, n), rho_bar)
    np.fill_diagonal(F_corr, 1.0)

    # scale back to covariance space: F_ij = rho_bar * sigma_i * sigma_j
    F = F_corr * np.outer(std, std)
    return F, rho_bar

F, rho_bar = constant_correlation_target(cov)
print(f"Average pairwise correlation: {rho_bar:.4f}")
print()
print("Constant-correlation target F:")
print(np.round(F, 4))

Average pairwise correlation: 0.4274

Constant-correlation target F:
[[0.2395 0.0566 0.0513 0.1315 0.0652 0.1205 0.0632 0.1243 0.0623 0.1085]
 [0.0566 0.0732 0.0284 0.0727 0.0361 0.0666 0.0349 0.0687 0.0345 0.06  ]
 [0.0513 0.0284 0.0602 0.0659 0.0327 0.0604 0.0317 0.0623 0.0312 0.0544]
 [0.1315 0.0727 0.0659 0.395  0.0838 0.1548 0.0812 0.1596 0.08   0.1393]
 [0.0652 0.0361 0.0327 0.0838 0.0972 0.0768 0.0403 0.0792 0.0397 0.0691]
 [0.1205 0.0666 0.0604 0.1548 0.0768 0.3319 0.0744 0.1463 0.0734 0.1277]
 [0.0632 0.0349 0.0317 0.0812 0.0403 0.0744 0.0913 0.0767 0.0385 0.067 ]
 [0.1243 0.0687 0.0623 0.1596 0.0792 0.1463 0.0767 0.353  0.0757 0.1317]
 [0.0623 0.0345 0.0312 0.08   0.0397 0.0734 0.0385 0.0757 0.0888 0.066 ]
 [0.1085 0.06   0.0544 0.1393 0.0691 0.1277 0.067  0.1317 0.066  0.2689]]


3. **Single-factor (market model)** $F = \beta\beta^T\sigma_m^2 + D$ — the CAPM/single-index covariance: every asset loads on one market factor, with diagonal idiosyncratic variances $D$.

In [5]:
def single_factor_target(returns_df, market_ticker="QQQ"):
    """
    Single-factor (market model) shrinkage target.

        F = beta @ beta.T * sigma_m^2 + diag(D)

    where:
        beta_i  = cov(r_i, r_m) / var(r_m)      (OLS beta on the market)
        sigma_m^2 = annualized market variance
        D_i     = var(r_i) - beta_i^2 * sigma_m^2  (idiosyncratic variance)

    All quantities are annualized (x252) to match a covariance matrix computed
    from daily returns.
    """
    # download market returns and align to the same trading days
    r_m = get_returns(market_ticker, start="2024-01-01")

    aligned = returns_df.join(r_m.rename(market_ticker), how="inner")
    R  = aligned[returns_df.columns].values   # (T, n) asset returns
    rm = aligned[market_ticker].values        # (T,)  market returns

    T = len(rm)
    sigma_m2 = rm.var(ddof=1) * 252                          # annualized market variance

    # cov(r_i, r_m) for each asset in one matrix multiply
    R_dm  = R  - R.mean(axis=0)                              # demeaned assets  (T, n)
    rm_dm = rm - rm.mean()                                   # demeaned market  (T,)
    cov_im = (R_dm.T @ rm_dm) / (T - 1) * 252               # (n,)  annualized

    beta = cov_im / sigma_m2                                  # (n,)
    var_assets = R.var(axis=0, ddof=1) * 252                 # (n,)  annualized

    # idiosyncratic variance: residual after stripping market exposure
    # clip to 0 to guard against tiny negative values from sampling noise
    D = np.maximum(var_assets - beta**2 * sigma_m2, 0)       # (n,)

    F = np.outer(beta, beta) * sigma_m2 + np.diag(D)
    return F, beta, sigma_m2

F_sf, beta, sigma_m2 = single_factor_target(returns, market_ticker="QQQ")

print(f"Market variance (annualised): {sigma_m2:.4f}  (σ_m = {sigma_m2**0.5:.2%})")
print(f"\nBetas vs QQQ:  {dict(zip(tickers, beta.round(3)))}")
print("\nSingle-factor target F (annualised covariance):")
print(np.round(F_sf, 4))

Market variance (annualised): 0.0424  (σ_m = 20.58%)

Betas vs QQQ:  {'NVDA': np.float64(1.779), 'AAPL': np.float64(0.832), 'MSFT': np.float64(0.772), 'MU': np.float64(2.032), 'AMZN': np.float64(1.095), 'AMD': np.float64(1.894), 'GOOGL': np.float64(0.893), 'TSLA': np.float64(1.792), 'GOOG': np.float64(0.883), 'AVGO': np.float64(1.818)}

Single-factor target F (annualised covariance):
[[0.2395 0.0627 0.0582 0.1532 0.0825 0.1428 0.0673 0.1351 0.0666 0.137 ]
 [0.0627 0.0732 0.0272 0.0716 0.0386 0.0668 0.0315 0.0632 0.0311 0.0641]
 [0.0582 0.0272 0.0602 0.0664 0.0358 0.0619 0.0292 0.0586 0.0289 0.0594]
 [0.1532 0.0716 0.0664 0.395  0.0943 0.1631 0.0769 0.1543 0.0761 0.1565]
 [0.0825 0.0386 0.0358 0.0943 0.0972 0.0879 0.0414 0.0831 0.041  0.0844]
 [0.1428 0.0668 0.0619 0.1631 0.0879 0.3319 0.0716 0.1438 0.0709 0.1459]
 [0.0673 0.0315 0.0292 0.0769 0.0414 0.0716 0.0913 0.0678 0.0334 0.0688]
 [0.1351 0.0632 0.0586 0.1543 0.0831 0.1438 0.0678 0.353  0.0671 0.138 ]
 [0.0666 0.0311 0.0289 0.0761

## Ledoit & Wolf

Ledoit and Wolf's celebrated contribution is an **analytical, data-driven optimal $\delta$**, computed in closed form with no cross-validation, no tuning, no simulation.

**The objective.** Choose $\delta$ to minimize the expected distance between the shrunk estimate and the *true* covariance, measured in squared Frobenius norm:

$$
\delta^* = \arg\min_\delta \ \mathbb{E}\left[\|\hat{\Sigma}_{\text{shrink}}(\delta) - \Sigma_{\text{true}}\|_F^2\right]
$$

This is a pure bias-variance optimization: $\delta$ too small keeps the sample matrix's variance; $\delta$ too large injects too much bias from the target. The minimizer balances the two.

**The result.** The optimal intensity has the form:

$$
\delta^* = \frac{1}{T}\cdot\frac{\sum \text{(variances of the sample covariance entries)}}{\sum \text{(squared deviations of sample covariance from target)}} = \frac{\pi - \rho}{\gamma}
$$

where, in Ledoit-Wolf's notation: $\pi$ measures the **total estimation variance** of the sample covariance entries (how noisy $\hat\Sigma$ is), $\rho$ captures the **covariance between the estimation errors** of the sample matrix and the target, and $\gamma$ measures the **misspecification** of the target (how far $F$ is from the true $\Sigma$). All three are estimated directly from the data.

Read the structure intuitively — this is the payoff:

- **Numerator $\pi - \rho$ (roughly, how noisy the sample estimate is):** more estimation noise → shrink more. With little data (small $T$, large $\pi$), $\delta^*$ rises toward 1 — lean on the target.
- **Denominator $\gamma$ (how wrong the target is):** the more misspecified the target, the *less* you shrink toward it. A badly-chosen target → small $\delta^*$.
- **The $1/T$ scaling:** more data → less shrinkage. As $T \to \infty$, $\delta^* \to 0$ and you recover the pure sample covariance, which is consistent in that limit.

$\rho$ is the correction for the target being estimated from the same data as the sample covariance — its diagonal reuses $\pi_{ii}$, and its off-diagonal $\vartheta$ terms measure how the sample variances (which build the target) co-vary with the sample covariances.

### Scaled Identity

For the scaled-identity target, $\rho$ collapses dramatically. The identity target is built from just *one* data-derived scalar — $\mu = \text{tr}(S)/N$, the average variance. There are no data-derived off-diagonal entries at all (the off-diagonals of $F$ are exactly zero, fixed, non-random).

**The result (Ledoit-Wolf 2004, "A well-conditioned estimator").** In their formulation the optimal shrinkage is written slightly differently — as a ratio of $\beta^2$ over $\delta^2$ rather than $(\pi-\rho)/\gamma$ — and the $\rho$-analog term is asymptotically negligible and dropped entirely. Their four quantities:

$$
\bar\mu = \frac{\text{tr}(S)}{N}, \qquad \alpha^2 = |S - \bar\mu I|_F^2 \quad(\text{the } \gamma\text{-analog: target misfit})
$$

$$
\beta^2 = \frac{1}{T^2}\sum_{t=1}^T |x_t x_t^T - S|_F^2 \quad(\text{the } \pi\text{-analog: total estimation noise})
$$

with $\bar\beta^2 = \min(\beta^2, \alpha^2)$ (a clamp), and the optimal intensity:

$$
\delta^* = \frac{\bar\beta^2}{\alpha^2 + \bar\beta^2} = \frac{\bar\beta^2}{|S|_F^2 / \dots}
$$

— more cleanly, $\delta^* = \bar\beta^2 / \zeta^2$ where $\zeta^2 = \alpha^2 + \bar\beta^2 = |S|_F^2 - N\bar\mu^2 + \bar\beta^2$. You compute:

- $\bar\mu$ — average of the diagonal (one trace).
- $\alpha^2$ (misfit) — squared Frobenius distance from $S$ to $\bar\mu I$, which is $|S|_F^2 - N\bar\mu^2$.
- $\beta^2$ (noise) — the average squared Frobenius distance between each period's outer product $x_t x_t^T$ and $S$. This is the identity-target analog of $\pi$, computed in one pass over the $T$ observations.

Then $\delta^* = \bar\beta^2/\zeta^2$, clamped to $[0,1]$.

In [6]:
def ledoit_wolf_identity(returns):
    """
    Ledoit-Wolf optimal shrinkage toward the scaled-identity target
    (Ledoit & Wolf 2004, "A well-conditioned estimator for large-dimensional
    covariance matrices").

        Sigma_shrink = delta * F + (1 - delta) * S,   F = mu * I,  mu = tr(S)/N

    The scaled-identity target has no data-derived off-diagonals, so the rho
    cross-term vanishes and delta reduces to a clean ratio of two quantities:
    estimation noise (beta^2) over total dispersion (alpha^2 + beta^2).

    Parameters
    ----------
    returns : (T, N) array of asset returns. T = observations, N = assets.

    Returns
    -------
    sigma_shrink : (N, N) shrunk covariance matrix (always PSD / well-conditioned)
    delta        : the optimal shrinkage intensity in [0, 1]
    diagnostics  : dict with mu, alpha2, beta2 for inspection
    """
    X = np.asarray(returns, dtype=float)
    T, N = X.shape
    if T < 2:
        raise ValueError("need at least 2 observations")

    # Demean each column (asset). LW asymptotics use 1/T, not 1/(T-1).
    X = X - X.mean(axis=0, keepdims=True)

    # Sample covariance (1/T convention)
    S = (X.T @ X) / T                              # (N, N)

    # Target F = mu * I,  mu = average sample variance = tr(S)/N
    mu = np.trace(S) / N
    F = mu * np.eye(N)

    # alpha^2 = ||S - F||_F^2   (target misfit / dispersion of S around the target)
    alpha2 = np.linalg.norm(S - F, "fro") ** 2

    # beta_bar^2 = total estimation noise of S, estimated from the per-period scatter.
    # beta2 = (1/T^2) * sum_t || x_t x_t^T - S ||_F^2
    # Vectorized: for each period t, the outer product x_t x_t^T minus S.
    beta2 = 0.0
    for t in range(T):
        xt = X[t][:, None]                         # (N, 1)
        diff = xt @ xt.T - S                       # (N, N)
        beta2 += np.linalg.norm(diff, "fro") ** 2
    beta2 /= T ** 2

    # LW clamp: the noise term can't exceed the total dispersion
    beta2 = min(beta2, alpha2)

    # Optimal shrinkage intensity:  delta = beta^2 / (alpha^2 + beta^2)
    #   = (noise you can escape) / (noise + misfit)
    denom = alpha2 + beta2
    delta = 0.0 if denom == 0 else beta2 / denom
    delta = float(np.clip(delta, 0.0, 1.0))

    sigma_shrink = delta * F + (1.0 - delta) * S

    diagnostics = {"mu": mu, "alpha2": alpha2, "beta2": beta2}
    return sigma_shrink, delta, diagnostics

sigma_shrink, delta, diagnostics = ledoit_wolf_identity(returns)
print(f"Condition Number: {np.linalg.cond(sigma_shrink)}")

sigma_sk, delta_sk = ledoit_wolf(returns, assume_centered=False) # validate against sklearn, which implements this exact estimator
print(f"Delta: {delta:.4f}, Sklearn Delta: {delta_sk:.4f}")

Condition Number: 186.4039887438672
Delta: 0.0264, Sklearn Delta: 0.0271


### Constant-Correlation Target

The target F is *built from the same data* as the sample covariance: $f_{ij} = \bar{r}\sqrt{s_{ii}s_{jj}}$ depends on the sample variances $s_{ii}$, $s_{jj}$ and the average sample correlation $\bar r$. So the target's estimation error is *correlated* with the sample covariance's estimation error — they share the same noisy data. $\rho$ exists precisely to account for that dependence. (If the target were a fixed, non-data-derived matrix, this coupling would largely vanish.)

**The estimator**, splitting into diagonal and off-diagonal parts. With demeaned returns and $s_{ij} = \frac{1}{T}\sum_t (x_{it}-\bar x_i)(x_{jt}-\bar x_j)$:

$$
\hat\rho = \underbrace{\sum_{i=1}^N \hat\pi_{ii}}_{\text{diagonal}} + \underbrace{\sum_{i=1}^N\sum_{\substack{j=1\\ j\neq i}}^N \frac{\bar r}{2}\left(\sqrt{\tfrac{s_{jj}}{s_{ii}}}\,\hat\vartheta_{ii,ij} + \sqrt{\tfrac{s_{ii}}{s_{jj}}}\,\hat\vartheta_{jj,ij}\right)}_{\text{off-diagonal}}
$$

The two pieces:

**Diagonal** — $\sum_i \hat\pi_{ii}$, where $\hat\pi_{ii} = \frac{1}{T}\sum_t\{(x_{it}-\bar x_i)^2 - s_{ii}\}^2$. On the diagonal the target equals the sample variance $f_{ii}=s_{ii}$, so the covariance of the sample entry with the target entry is just the *variance* of $s_{ii}$ — which is exactly $\pi_{ii}$. That's why the diagonal of $\rho$ reuses the diagonal of $\pi$.

**Off-diagonal** — the $\vartheta$ cross-moments capture the covariance between the sample *variances* (which feed the target) and the sample *covariance*:

$$
\hat\vartheta_{ii,ij} = \frac{1}{T}\sum_{t=1}^T \{(x_{it}-\bar x_i)^2 - s_{ii}\}\{(x_{it}-\bar x_i)(x_{jt}-\bar x_j) - s_{ij}\}
$$

and symmetrically $\hat\vartheta_{jj,ij}$ with $j$ in the squared term. Each $\vartheta$ is a sample average of (variance-deviation × covariance-deviation) — literally the empirical covariance between "how $s_{ii}$ deviates from its mean" and "how $s_{ij}$ deviates from its mean." The $\bar r / 2$ and the $\sqrt{s_{jj}/s_{ii}}$ scalings come from differentiating the target $f_{ij}=\bar r\sqrt{s_{ii}s_{jj}}$ with respect to $s_{ii}$ and $s_{jj}$ (chain rule on the square root — that's where the half and the variance-ratio factors originate).

In [7]:
def ledoit_wolf_constant_correlation(returns):
    """
    Ledoit-Wolf optimal shrinkage toward the CONSTANT-CORRELATION target
    (Ledoit & Wolf 2004, "Honey, I Shrunk the Sample Covariance Matrix").

        Sigma_shrink = delta * F + (1 - delta) * S

    Target F: keeps each asset's own sample variance, replaces every pairwise
    correlation with the average sample correlation r_bar.
        F_ii = s_ii
        F_ij = r_bar * sqrt(s_ii * s_jj)   (i != j)

    Optimal intensity: delta = (pi - rho) / gamma, clamped to [0, 1].
      pi    = total estimation variance of the sample covariance entries
      gamma = ||F - S||_F^2  (target misfit)
      rho   = covariance between estimation errors of S and of the target F.
              Nonzero here because F is built from the same data (the s_ii and
              r_bar). Diagonal of rho reuses pi_ii; off-diagonal needs the
              theta cross-moments.

    Parameters
    ----------
    returns : (T, N) array of asset returns.

    Returns
    -------
    sigma_shrink : (N, N) shrunk covariance matrix
    delta        : optimal shrinkage intensity in [0, 1]
    diagnostics  : dict with pi, rho, gamma, r_bar
    """
    X = np.asarray(returns, dtype=float)
    T, N = X.shape
    if T < 2:
        raise ValueError("need at least 2 observations")

    # Demean (LW asymptotics use 1/T)
    X = X - X.mean(axis=0, keepdims=True)

    # Sample covariance, variances, std devs, correlations
    S = (X.T @ X) / T                       # (N, N)
    var = np.diag(S)                        # (N,)
    std = np.sqrt(var)                      # (N,)
    # average off-diagonal sample correlation r_bar
    R = S / np.outer(std, std)              # sample correlation matrix
    r_bar = (R.sum() - N) / (N * (N - 1))   # mean of off-diagonal entries

    # Constant-correlation target F
    F = r_bar * np.outer(std, std)
    np.fill_diagonal(F, var)                # diagonal keeps sample variance

    # ---- gamma: target misfit ||F - S||_F^2 ----
    gamma = np.linalg.norm(F - S, "fro") ** 2

    # ---- pi: total estimation variance of S entries ----
    # pi_ij = (1/T) sum_t [ (x_it x_jt) - s_ij ]^2
    # Build the per-period products tensor implicitly.
    # Y[t] = outer(x_t, x_t); pi_ij = mean_t (Y[t]_ij - S_ij)^2
    # Vectorized: pi_ij = mean_t (x_it x_jt)^2 - S_ij^2
    # because mean_t (x_it x_jt) = s_ij.
    X2 = X ** 2
    pi_mat = (X2.T @ X2) / T - S ** 2       # (N, N), the pi_ij matrix
    pi = pi_mat.sum()

    # ---- rho: diagonal part + off-diagonal theta cross-moments ----
    # Diagonal: sum_i pi_ii
    rho_diag = np.trace(pi_mat)

    # Off-diagonal: (r_bar/2) * sum_{i!=j} [ sqrt(s_jj/s_ii) theta_ii,ij
    #                                       + sqrt(s_ii/s_jj) theta_jj,ij ]
    # theta_kk,ij = (1/T) sum_t [ (x_kt^2 - s_kk)(x_it x_jt - s_ij) ]
    #
    # Compute the term matrix without an explicit t-loop.
    # Let A_t = X (demeaned). Define:
    #   E_kk,t   = x_kt^2 - s_kk          (deviation of variance estimate)
    #   E_ij,t   = x_it x_jt - s_ij       (deviation of covariance estimate)
    # theta_ii,ij = mean_t E_ii,t * E_ij,t
    #
    # We need, for each (i,j), theta_ii,ij and theta_jj,ij.
    # theta_ii,ij = (1/T) sum_t (x_it^2 - s_ii)(x_it x_jt - s_ij)
    #             = (1/T) sum_t x_it^3 x_jt  - s_ii s_ij        (after expansion)
    # Use the closed expansion: mean_t[x_it^2 x_it x_jt] - s_ii s_ij
    #   = mean_t[x_it^3 x_jt] - s_ii s_ij
    # Build M3[i,j] = mean_t (x_it^3 * x_jt)
    X3 = X ** 3
    M3 = (X3.T @ X) / T                     # M3[i,j] = mean_t x_it^3 x_jt
    # theta_ii,ij = M3[i,j] - var[i]*S[i,j]
    theta_ii = M3 - np.outer(var, np.ones(N)) * S      # (N,N): theta for (ii,ij)
    # theta_jj,ij = mean_t x_jt^3 x_it - s_jj s_ij = M3[j,i] - var[j]*S[i,j]
    theta_jj = M3.T - np.outer(np.ones(N), var) * S    # (N,N): theta for (jj,ij)

    # scaling factors sqrt(s_jj/s_ii) and sqrt(s_ii/s_jj)
    ratio = np.outer(1.0 / std, std)        # ratio[i,j] = std_j / std_i = sqrt(s_jj/s_ii)
    term = ratio * theta_ii + (1.0 / ratio) * theta_jj
    # zero the diagonal (off-diagonal sum only)
    np.fill_diagonal(term, 0.0)
    rho_off = (r_bar / 2.0) * term.sum()

    rho = rho_diag + rho_off

    # ---- optimal shrinkage intensity ----
    kappa = (pi - rho) / gamma
    delta = float(np.clip(kappa / T, 0.0, 1.0))

    sigma_shrink = delta * F + (1.0 - delta) * S

    diagnostics = {"pi": pi, "rho": rho, "gamma": gamma, "r_bar": r_bar,
                   "kappa": kappa, "delta_raw": kappa / T}
    return sigma_shrink, delta, diagnostics
sigma_shrink, delta, diagnostics = ledoit_wolf_constant_correlation(returns)
print(f"Condition Number: {np.linalg.cond(sigma_shrink)}")

Condition Number: 65.48745741291923


So $\delta^*$ automatically shrinks **more** when data is scarce or the sample is noisy, and **less** when data is abundant or the target is badly specified. It adapts to your actual situation with no knobs to turn — which is exactly why Ledoit-Wolf became the practitioner standard. You don't guess the shrinkage; the data tells you.

**Why it won in practice:**

- **No tuning** — $\delta^*$ is computed, not cross-validated. One pass over the data.
- **Always valid** — produces a PSD, invertible, well-conditioned matrix even when $n > T$.
- **Provably optimal** (for the chosen target, under the Frobenius loss) — it's not a heuristic; it's the variance-minimizing blend.
- **Cheap** — closed-form, scales to thousands of assets.